In [1]:
# ==============================================================================
# 🛠️ RADIOLOGICAL WORKSTATION V20: FILTRO DE MEDIANA Y ANÁLISIS ESPACIAL
# ==============================================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.morphology import skeletonize
from skimage.util import img_as_ubyte
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# --- FUNCIONES DE DIBUJO PERSONALIZADAS (DASHED RED) ---

def draw_dashed_line(img, pt1, pt2, color, thickness=1, dash_length=10):
    dist = np.sqrt((pt1[0]-pt2[0])**2 + (pt1[1]-pt2[1])**2)
    dashes = int(dist / dash_length)
    for i in range(dashes):
        start = [int(pt1[0] + (pt2[0]-pt1[0]) * i / dashes), int(pt1[1] + (pt2[1]-pt1[1]) * i / dashes)]
        end = [int(pt1[0] + (pt2[0]-pt1[0]) * (i + 0.5) / dashes), int(pt1[1] + (pt2[1]-pt1[1]) * (i + 0.5) / dashes)]
        cv2.line(img, tuple(start), tuple(end), color, thickness)

def draw_dashed_rect(img, pt1, pt2, color, thickness=1, dash_length=10):
    x1, y1 = pt1; x2, y2 = pt2
    draw_dashed_line(img, (x1, y1), (x2, y1), color, thickness, dash_length)
    draw_dashed_line(img, (x2, y1), (x2, y2), color, thickness, dash_length)
    draw_dashed_line(img, (x2, y2), (x1, y2), color, thickness, dash_length)
    draw_dashed_line(img, (x1, y2), (x1, y1), color, thickness, dash_length)

def draw_dashed_ellipse(img, center, axes, color, thickness=1, dash_length=5):
    for i in range(0, 360, dash_length * 2):
        cv2.ellipse(img, center, axes, 0, i, i + dash_length, color, thickness)

# --- LÓGICA DE ARCHIVOS ---
def obtener_lista_imagenes():
    extensiones = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')
    archivos = [f for f in os.listdir('.') if f.lower().endswith(extensiones)]
    return sorted(archivos) if archivos else ["No se encontraron imágenes"]

# --- FUNCIONES MATEMÁTICAS ---

def calcular_metricas_v20(orig, proc, mask, es_global):
    g_orig = cv2.cvtColor(orig, cv2.COLOR_BGR2GRAY).astype(float)
    g_proc = cv2.cvtColor(proc, cv2.COLOR_BGR2GRAY).astype(float)
    if es_global:
        mse = np.mean((g_orig - g_proc)**2)
    else:
        if np.sum(mask) == 0: return 0, 0, 0
        mse = np.mean((g_orig[mask>0] - g_proc[mask>0])**2)
    psnr = 100 if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))
    score, _ = ssim(g_orig.astype(np.uint8), g_proc.astype(np.uint8), full=True, data_range=255)
    return mse, psnr, score

def aplicar_filtro_simetria(img, orient, pos, metodo):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    if orient == 'Vertical':
        pos = np.clip(pos, 1, w-1); h1, h2 = gray[:, :pos], gray[:, pos:]
    else:
        pos = np.clip(pos, 1, h-1); h1, h2 = gray[:pos, :], gray[pos:, :]
    hist1 = cv2.calcHist([h1], [0], None, [256], [0, 256]) / h1.size
    hist2 = cv2.calcHist([h2], [0], None, [256], [0, 256]) / h2.size
    diff = np.abs(hist1 - hist2).flatten()
    diff = diff / (np.max(diff) + 1e-7)
    if metodo == 'OTSU':
        _, diff_bin = cv2.threshold((diff * 255).astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        diff = diff_bin.flatten().astype(float) / 255.0
    lut = np.clip(np.arange(256) * (1.0 + diff * 1.5), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.LUT(gray, lut), cv2.COLOR_GRAY2BGR), diff

def get_transfer_function(dist, f_type, mode, d0, d_low, d_high, n):
    def base_h(d, cutoff):
        if f_type == 'Gaussiano': return np.exp(-(d**2)/(2*(cutoff**2)))
        if f_type == 'Butterworth': return 1/(1 + (d/cutoff)**(2*n))
        if f_type == 'Chebyshev': return 1/np.sqrt(1 + (d/cutoff)**(2*n))
        return np.ones_like(d)
    if mode == 'PasaBajo': return base_h(dist, d0)
    if mode == 'PasaAlto': return 1 - base_h(dist, d0)
    if mode == 'PasaBanda': return base_h(dist, d_high) * (1 - base_h(dist, d_low))
    return np.ones_like(dist)

def aplicar_pipeline_v20(img, roi_p, spat_p, sym_p, freq_p, morph_p):
    orig = img.copy(); h, w = img.shape[:2]
    if roi_p['shape'] == 'Ninguna': mask = np.full((h, w), 255, dtype=np.uint8)
    else:
        mask = np.zeros((h, w), dtype=np.uint8)
        rx, ry, rw, rh = roi_p['x'], roi_p['y'], roi_p['w'], roi_p['h']
        if roi_p['shape'] == 'Rectángulo': cv2.rectangle(mask, (rx, ry), (min(rx+rw, w), min(ry+rh, h)), 255, -1)
        elif roi_p['shape'] == 'Círculo': cv2.circle(mask, (rx+rw//2, ry+rw//2), rw//2, 255, -1)
        elif roi_p['shape'] == 'Óvalo': cv2.ellipse(mask, (rx+rw//2, ry+rh//2), (rw//2, rh//2), 0, 0, 360, 255, -1)

    def blend(p, o):
        return cv2.add(cv2.bitwise_and(p, p, mask=mask), cv2.bitwise_and(o, o, mask=cv2.bitwise_not(mask)))

    img_spat = orig.copy()
    if spat_p['type'] != 'Ninguno':
        lab = cv2.cvtColor(img_spat, cv2.COLOR_BGR2LAB); l, a, b = cv2.split(lab)
        if spat_p['type'] == 'Logaritmica':
            c = (255 / np.log(1 + 255)) * spat_p['log_g']
            l = np.clip(c * np.log(1 + l.astype(float)), 0, 255).astype(np.uint8)
        elif spat_p['type'] == 'Ecualizacion': l = cv2.equalizeHist(l)
        elif spat_p['type'] == 'CLAHE':
            l = cv2.createCLAHE(clipLimit=spat_p['c_c'], tileGridSize=(spat_p['c_g'], spat_p['c_g'])).apply(l)
        elif spat_p['type'] == 'Mediana':
            k_size = spat_p['med_k']
            if k_size % 2 == 0: k_size += 1 # Asegurar impar
            l = cv2.medianBlur(l, k_size)
        img_spat = blend(cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR), orig)

    img_sym = img_spat.copy(); sym_diff = None
    if sym_p['active']:
        full_sym, sym_diff = aplicar_filtro_simetria(img_spat, sym_p['orient'], sym_p['pos'], sym_p['metodo'])
        img_sym = blend(full_sym, img_spat)

    img_freq = img_sym.copy()
    if freq_p['type'] != 'Ninguno':
        res_f = []
        for ch in cv2.split(img_freq):
            f = np.fft.fftshift(np.fft.fft2(ch.astype(float)))
            y, x = np.ogrid[:h, :w]; D = np.sqrt((x - w//2)**2 + (y - h//2)**2) + 1e-8
            H = get_transfer_function(D, freq_p['type'], freq_p['mode'], freq_p['d0'], freq_p['d_low'], freq_p['d_high'], freq_p['n'])
            res_f.append(np.clip(np.abs(np.fft.ifft2(np.fft.ifftshift(f*H))), 0, 255).astype(np.uint8))
        img_freq = blend(cv2.merge(res_f), img_sym)

    img_final = img_freq.copy()
    if morph_p != 'Ninguno':
        if morph_p == 'Deteccion_Bordes_Binario':
            full_m = cv2.cvtColor(cv2.Canny(cv2.cvtColor(img_final, cv2.COLOR_BGR2GRAY), 100, 200), cv2.COLOR_GRAY2BGR)
        elif morph_p == 'Realce_Bordes_Gradiente':
            g = cv2.cvtColor(img_final, cv2.COLOR_BGR2GRAY); gx = cv2.Sobel(g, cv2.CV_64F, 1, 0); gy = cv2.Sobel(g, cv2.CV_64F, 0, 1)
            full_m = cv2.cvtColor(cv2.convertScaleAbs(cv2.magnitude(gx, gy)), cv2.COLOR_GRAY2BGR)
        elif morph_p == 'Esqueletizacion':
            _, b = cv2.threshold(cv2.cvtColor(img_final, cv2.COLOR_BGR2GRAY), 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
            full_m = cv2.cvtColor(img_as_ubyte(skeletonize(b>0)), cv2.COLOR_GRAY2BGR)
        img_final = blend(full_m, img_freq)
    
    return orig, img_spat, img_sym, img_freq, img_final, mask, sym_diff

# --- UI COMPONENTS ---
lista_imgs = obtener_lista_imagenes()
file_dropdown = widgets.Dropdown(options=lista_imgs, description='📁 Imágenes:')
gray_chk = widgets.Checkbox(value=False, description='Convertir a Grises', indent=False)
raw_preview_out = widgets.Output()

roi_sel = widgets.Dropdown(options=['Ninguna', 'Rectángulo', 'Círculo', 'Óvalo'], description='ROI:')
roi_ctrls = [widgets.IntSlider(description=n, min=0, max=1000, value=100) for n in ['X', 'Y', 'W', 'H']]

# Añadido Mediana al dropdown y slider med_k
spat_sel = widgets.Dropdown(options=['Ninguno', 'Logaritmica', 'Ecualizacion', 'CLAHE', 'Mediana'], description='Espacial:')
spat_ctrls = [widgets.FloatSlider(description='Clip', min=0.1, max=10, value=3), 
              widgets.IntSlider(description='Grid', min=1, max=32, value=8),
              widgets.FloatSlider(description='Log Gain', min=0.1, max=3, value=1),
              widgets.IntSlider(description='Median K', min=3, max=15, step=2, value=3)]

sym_active = widgets.Checkbox(value=False, description='Activar Simetría')
sym_orient = widgets.Dropdown(options=['Vertical', 'Horizontal'], description='Eje:')
sym_pos = widgets.IntSlider(description='Posición:', min=0, max=1000, value=250)
sym_metodo = widgets.Dropdown(options=['Diferencia', 'OTSU'], description='Método:')

freq_sel = widgets.Dropdown(options=['Ninguno', 'Gaussiano', 'Butterworth', 'Chebyshev'], description='Filtro:')
freq_mod = widgets.Dropdown(options=['PasaBajo', 'PasaAlto', 'PasaBanda'], description='Modo:')
freq_ctrls = [widgets.IntSlider(description='D0/High', min=1, max=500, value=50),
              widgets.IntSlider(description='D Low', min=1, max=500, value=20),
              widgets.IntSlider(description='n (Orden)', min=1, max=10, value=2)]

morph_sel = widgets.Dropdown(options=['Ninguno', 'Deteccion_Bordes_Binario', 'Realce_Bordes_Gradiente', 'Esqueletizacion'], description='Morf:')

roi_preview_out = widgets.Output(); sym_preview_out = widgets.Output(); graph_out = widgets.Output(); main_out = widgets.Output()

col1 = widgets.VBox([widgets.HTML("<b>📍 ROI & ESPACIAL</b>"), roi_sel, *roi_ctrls, spat_sel, *spat_ctrls, widgets.HTML("<br><b>🎯 UBICACIÓN ROI</b>"), roi_preview_out])
col2 = widgets.VBox([widgets.HTML("<b>⚖️ SIMETRÍA</b>"), sym_active, sym_orient, sym_pos, sym_metodo, widgets.HTML("<br><b>📡 FRECUENCIA</b>"), freq_sel, freq_mod, *freq_ctrls])
col3 = widgets.VBox([widgets.HTML("<b>🦴 MORFOLOGÍA</b>"), morph_sel, widgets.HTML("<br><b>✂️ EJE SIMETRÍA</b>"), sym_preview_out])

header = widgets.HBox([file_dropdown, gray_chk, raw_preview_out], layout=widgets.Layout(align_items='center', border='2px solid #3498db', padding='10px'))
controls_ui = widgets.HBox([col1, col2, col3, graph_out], layout=widgets.Layout(border='1px solid gray', padding='10px'))

def update(*args):
    with main_out:
        clear_output(wait=True)
        img_path = file_dropdown.value
        if not img_path or img_path == "No se encontraron imágenes": return
        img_raw = cv2.imread(img_path)
        if img_raw is None: return
        
        is_color = (len(img_raw.shape) == 3 and not (np.all(img_raw[:,:,0] == img_raw[:,:,1]) and np.all(img_raw[:,:,1] == img_raw[:,:,2])))
        gray_chk.layout.display = 'block' if is_color else 'none'
        img = cv2.cvtColor(cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR) if gray_chk.value else img_raw.copy()

        h, w = img.shape[:2]
        roi_ctrls[0].max, roi_ctrls[2].max = w, w; roi_ctrls[1].max, roi_ctrls[3].max = h, h
        sym_pos.max = w if sym_orient.value == 'Vertical' else h
        freq_ctrls[1].disabled = (freq_mod.value != 'PasaBanda')
        for widget in roi_ctrls: widget.disabled = (roi_sel.value == 'Ninguna')
        
        # Visibilidad de sliders espaciales
        spat_ctrls[0].disabled = spat_ctrls[1].disabled = (spat_sel.value != 'CLAHE')
        spat_ctrls[2].disabled = (spat_sel.value != 'Logaritmica')
        spat_ctrls[3].disabled = (spat_sel.value != 'Mediana')
        
        rp = {'shape': roi_sel.value, 'x': roi_ctrls[0].value, 'y': roi_ctrls[1].value, 'w': roi_ctrls[2].value, 'h': roi_ctrls[3].value}
        sp = {'type': spat_sel.value, 'c_c': spat_ctrls[0].value, 'c_g': spat_ctrls[1].value, 'log_g': spat_ctrls[2].value, 'med_k': spat_ctrls[3].value}
        syp = {'active': sym_active.value, 'orient': sym_orient.value, 'pos': sym_pos.value, 'metodo': sym_metodo.value}
        fp = {'type': freq_sel.value, 'mode': freq_mod.value, 'd0': freq_ctrls[0].value, 'd_high': freq_ctrls[0].value, 'd_low': freq_ctrls[1].value, 'n': freq_ctrls[2].value}
        
        i_orig, i_spat, i_sym, i_freq, i_final, mask, sym_curve = aplicar_pipeline_v20(img, rp, sp, syp, fp, morph_sel.value)
        es_global = (roi_sel.value == 'Ninguna')

        with raw_preview_out:
            clear_output(wait=True); plt.figure(figsize=(1.5, 1.5)); plt.imshow(cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
        
        with roi_preview_out:
            clear_output(wait=True); prev = img.copy()
            if not es_global:
                rx, ry, rw, rh = rp['x'], rp['y'], rp['w'], rp['h']
                color_red = (0, 0, 255)
                if rp['shape'] == 'Rectángulo': draw_dashed_rect(prev, (rx, ry), (min(rx+rw, w), min(ry+rh, h)), color_red, 2)
                elif rp['shape'] == 'Círculo': draw_dashed_ellipse(prev, (rx+rw//2, ry+rw//2), (rw//2, rw//2), color_red, 2)
                elif rp['shape'] == 'Óvalo': draw_dashed_ellipse(prev, (rx+rw//2, ry+rh//2), (rw//2, rh//2), color_red, 2)
            plt.figure(figsize=(2.5, 2.5)); plt.imshow(cv2.cvtColor(prev, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
        
        with sym_preview_out:
            clear_output(wait=True); prev_s = img.copy()
            color_red = (0, 0, 255)
            if sym_orient.value == 'Vertical': draw_dashed_line(prev_s, (sym_pos.value, 0), (sym_pos.value, h), color_red, 3)
            else: draw_dashed_line(prev_s, (0, sym_pos.value), (w, sym_pos.value), color_red, 3)
            plt.figure(figsize=(2.5, 2.5)); plt.imshow(cv2.cvtColor(prev_s, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

        with graph_out:
            clear_output(wait=True); fig, (axf, axs, axy) = plt.subplots(3, 1, figsize=(4, 10))
            if fp['type'] != 'Ninguno':
                dax = np.linspace(0, 500, 200); hax = get_transfer_function(dax, fp['type'], fp['mode'], fp['d0'], fp['d_low'], fp['d_high'], fp['n'])
                axf.plot(dax, hax, color='red', lw=2); axf.set_title("H(D) Frecuencia"); axf.grid(True, alpha=0.3)
            if sp['type'] == 'Logaritmica':
                r_v = np.linspace(0, 255, 256); c_l = (255 / np.log(1 + 255)) * sp['log_g']; s_v = np.clip(c_l * np.log(1 + r_v), 0, 255)
                axs.plot(r_v, s_v, color='green', lw=2); axs.set_title("Mapeo Logarítmico"); axs.grid(True, alpha=0.3)
            elif sp['type'] in ['Ecualizacion', 'CLAHE', 'Mediana']:
                axs.hist(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).ravel(), bins=256, range=[0,256], color='blue', alpha=0.4, label='Base')
                axs.hist(cv2.cvtColor(i_spat, cv2.COLOR_BGR2GRAY).ravel(), bins=256, range=[0,256], color='red', alpha=0.4, label='Spat')
                axs.set_title("Histogramas"); axs.legend(fontsize='xx-small'); axs.grid(True, alpha=0.2)
            if sym_curve is not None:
                axy.plot(sym_curve, color='orange', lw=2); axy.set_title("Curva de Asimetría"); axy.fill_between(range(256), sym_curve, alpha=0.3, color='orange')
            plt.tight_layout(); plt.show()

        fig, axes = plt.subplots(1, 5, figsize=(25, 6))
        titles = ["1. Base Pipeline", f"2. Espacial\n({spat_sel.value})", f"3. Simetría\n({sym_metodo.value})", f"4. Frecuencia\n({freq_sel.value})", f"5. Final\n({morph_sel.value})"]
        imgs = [i_orig, i_spat, i_sym, i_freq, i_final]
        for i, (ax, title, image) in enumerate(zip(axes, titles, imgs)):
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); ax.axis('off')
            if i > 0:
                m, p, s = calcular_metricas_v20(img, image, mask, es_global)
                ax.set_title(f"{title}\nMSE:{m:.1f} | PSNR:{p:.1f}dB", fontsize=9)
            else: ax.set_title(title, fontweight='bold')
        plt.tight_layout(); plt.show()

all_w = [file_dropdown, gray_chk, roi_sel, *roi_ctrls, spat_sel, *spat_ctrls, sym_active, sym_orient, sym_pos, sym_metodo, freq_sel, freq_mod, *freq_ctrls, morph_sel]
for w in all_w: w.observe(update, 'value')

display(header, controls_ui, main_out)
update()

Output()